# **Imports**

In [1]:
import warnings

warnings.filterwarnings('ignore')

In [10]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.metrics import mean_squared_error
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import save_model

# !pip install optuna
import optuna

# **Test A**

In [3]:
cd /content/drive/MyDrive/[Projects]/Dacon/운수종사자 인지적 특성 데이터를 활용한 교통사고 위험 예측 AI 경진대회/Data

/content/drive/MyDrive/[Projects]/Dacon/운수종사자 인지적 특성 데이터를 활용한 교통사고 위험 예측 AI 경진대회/Data


In [4]:
train_raw_a = pd.read_csv('./train/A.csv')
train_label = pd.read_csv('./train.csv')

In [5]:
train_label_a = train_label[train_label['Test'] == 'A']

train_a = train_label_a.merge(train_raw_a, on='Test_id', how='left')

train_a = train_a.drop('Test_y', axis=1)
train_a.rename(columns={'Test_x': 'Test'})
train_a = train_a.dropna()

train_a.info()

<class 'pandas.core.frame.DataFrame'>
Index: 647237 entries, 0 to 647240
Data columns (total 38 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Test_id     647237 non-null  object
 1   Test_x      647237 non-null  object
 2   Label       647237 non-null  int64 
 3   PrimaryKey  647237 non-null  object
 4   Age         647237 non-null  object
 5   TestDate    647237 non-null  int64 
 6   A1-1        647237 non-null  object
 7   A1-2        647237 non-null  object
 8   A1-3        647237 non-null  object
 9   A1-4        647237 non-null  object
 10  A2-1        647237 non-null  object
 11  A2-2        647237 non-null  object
 12  A2-3        647237 non-null  object
 13  A2-4        647237 non-null  object
 14  A3-1        647237 non-null  object
 15  A3-2        647237 non-null  object
 16  A3-3        647237 non-null  object
 17  A3-4        647237 non-null  object
 18  A3-5        647237 non-null  object
 19  A3-6        647237 non-null 

## **Features**

In [6]:
def safe_fromstring(x, dtype=float):
    if isinstance(x, str) and x.strip():
        return np.fromstring(x, sep=',', dtype=dtype)
    return np.array([], dtype=dtype)

def preprocess_A(train_A):
    df = train_A.copy()

    print("Step 1: Age 파생...")
    df["Age"] = df["Age"].astype(str).str.extract(r'(\d+)').astype(float)
    feats = pd.DataFrame(index=df.index)

    print("Step 2: Sequence 변환...")
    seq_cols = [
        'A1-1', 'A1-2', 'A1-3', 'A1-4',
        'A2-1', 'A2-2', 'A2-3', 'A2-4',
        'A3-1', 'A3-2', 'A3-3', 'A3-4', 'A3-5', 'A3-6', 'A3-7',
        'A4-1', 'A4-2', 'A4-3', 'A4-4', 'A4-5',
        'A5-1', 'A5-2', 'A5-3'
    ]
    for col in seq_cols:
        feats[col + '_list'] = df[col].apply(lambda x: safe_fromstring(x, dtype=float))

    print("Step 3: A1 feature 생성...")
    a1_feats = feats.apply(compute_A1_features, axis=1)
    feats = pd.concat([feats, a1_feats], axis=1)

    print("Step 4: A2 feature 생성...")
    a2_feats = feats.apply(compute_A2_features, axis=1)
    feats = pd.concat([feats, a2_feats], axis=1)

    print("Step 5: A3 feature 생성...")
    a3_feats = feats.apply(compute_A3_features, axis=1)
    feats = pd.concat([feats, a3_feats], axis=1)

    print("Step 6: A4 feature 생성...")
    a4_feats = feats.apply(compute_A4_features, axis=1)
    feats = pd.concat([feats, a4_feats], axis=1)

    print("Step 7: A5 feature 생성...")
    a5_feats = feats.apply(compute_A5_features, axis=1)
    feats = pd.concat([feats, a5_feats], axis=1)

    print("Step 8: A6 feature 생성...")
    feats['A6_score'] = df['A6-1']
    feats['A6_zscore'] = (feats['A6_score'] - feats['A6_score'].mean()) / feats['A6_score'].std()

    print("Step 9: A7 feature 생성...")
    feats['A7_score'] = df['A7-1']
    feats['A7_zscore'] = (feats['A7_score'] - feats['A7_score'].mean()) / feats['A7_score'].std()

    print("Step 10: A8 feature 생성...")
    feats['A8_distortion_score'] = df['A8-1']
    feats['A8_consistency_score'] = df['A8-2']
    feats['A8_distortion_flag'] = (feats['A8_distortion_score'] > 5).astype(int)

    print("Step 9: A7 feature 생성...")
    feats['A9_emotional_stability'] = df['A9-1']
    feats['A9_behavior_stability'] = df['A9-2']
    feats['A9_reality_checking'] = df['A9-3']
    feats['A9_cognitive_agility'] = df['A9-4']
    feats['A9_stress_level'] = df['A9-5']
    feats['A9_total_score'] = feats[['A9_emotional_stability','A9_behavior_stability',
                                        'A9_reality_checking','A9_cognitive_agility','A9_stress_level']].sum(axis=1)
    feats['A9_stability_gap'] = feats['A9_emotional_stability'] - feats['A9_behavior_stability']

    feats = feats.fillna(0)

    print("A 검사 데이터 전처리 완료")
    list_cols = [f'{cols}_list' for cols in seq_cols]
    int_cols = [
        'A6-1', 'A7-1', 'A8-1', 'A8-2', 'A9-1', 'A9-2', 'A9-3', 'A9-4', 'A9-5'
    ]
    out = pd.concat([df.drop(columns=seq_cols + int_cols, errors="ignore"),
                     feats.drop(columns=list_cols, errors='ignore')], axis=1)
    return out

In [7]:
def compute_A1_features(row):
    # 리스트 추출
    d = row.get('A1-1_list', np.array([]))
    s = row.get('A1-2_list', np.array([]))
    r = row.get('A1-3_list', np.array([]))
    rt = row.get('A1-4_list', np.array([]))

    L = min(len(d), len(s), len(r), len(rt))
    if L == 0:
        return pd.Series({
            'A1_response_rate': 0,
            'A1_left_response_rate': 0,
            'A1_right_response_rate': 0,
            'A1_fast_response_rate': 0,
            'A1_mean_response_time': np.nan,
            'A1_fast_avg_rt': np.nan,
            'A1_direction_diff_rt': np.nan
        })

    d, s, r, rt = np.array(d[:L]), np.array(s[:L]), np.array(r[:L]), np.array(rt[:L])

    # 전체 응답률
    A1_response_rate = r.mean()

    # 왼쪽/오른쪽 조건 응답률
    A1_left_response_rate = r[d == 1].mean() if np.any(d == 1) else 0
    A1_right_response_rate = r[d == 2].mean() if np.any(d == 2) else 0

    # 빠름 조건(3)의 응답률
    A1_fast_response_rate = r[s == 3].mean() if np.any(s == 3) else 0

    # 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r == 1]
    A1_mean_response_time = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    # 빠름 조건에서의 평균 반응시간
    fast_rt = rt[(s == 3) & (r == 1)]
    A1_fast_avg_rt = fast_rt.mean() if len(fast_rt) > 0 else np.nan

    # 방향별 반응시간 차이 (left - right)
    left_rt = rt[(d == 1) & (r == 1)]
    right_rt = rt[(d == 2) & (r == 1)]
    A1_direction_diff_rt = left_rt.mean() - right_rt.mean() if len(left_rt) > 0 and len(right_rt) > 0 else np.nan

    return pd.Series({
        'A1_response_rate': A1_response_rate,
        'A1_left_response_rate': A1_left_response_rate,
        'A1_right_response_rate': A1_right_response_rate,
        'A1_fast_response_rate': A1_fast_response_rate,
        'A1_mean_response_time': A1_mean_response_time,
        'A1_fast_avg_rt': A1_fast_avg_rt,
        'A1_direction_diff_rt': A1_direction_diff_rt
    })

def compute_A2_features(row):
    s1 = row.get('A2-1_list', np.array([]))
    s2 = row.get('A2-2_list', np.array([]))
    r  = row.get('A2-3_list', np.array([]))
    rt = row.get('A2-4_list', np.array([]))

    L = min(len(s1), len(s2), len(r), len(rt))
    if L == 0:
        return pd.Series({
            'A2_response_rate': 0,
            'A2_slow_to_fast_rt_diff': np.nan,
            'A2_correct_ratio_by_speed': np.nan,
            'A2_mean_response_time': np.nan
        })

    s1, s2, r, rt = np.array(s1[:L]), np.array(s2[:L]), np.array(r[:L]), np.array(rt[:L])

    # 전체 응답률
    A2_response_rate = r.mean()

    # 느림/빠름 인덱스 (가정: 1=느림, 2=빠름)
    slow_idx = np.where(s1 == 1)[0]
    fast_idx = np.where(s1 == 2)[0]

    # 느림→빠름 조건 반응시간 차이
    slow_rt = rt[slow_idx & (r[slow_idx]==1)] if len(slow_idx) > 0 else np.array([])
    fast_rt = rt[fast_idx & (r[fast_idx]==1)] if len(fast_idx) > 0 else np.array([])
    A2_slow_to_fast_rt_diff = fast_rt.mean() - slow_rt.mean() if len(slow_rt) > 0 and len(fast_rt) > 0 else np.nan

    # 속도 조건별 응답률 비교 (fast/slow)
    slow_resp = r[slow_idx].mean() if len(slow_idx) > 0 else np.nan
    fast_resp = r[fast_idx].mean() if len(fast_idx) > 0 else np.nan
    if not np.isnan(slow_resp) and not np.isnan(fast_resp) and slow_resp != 0:
        A2_correct_ratio_by_speed = fast_resp / slow_resp
    else:
        A2_correct_ratio_by_speed = np.nan

    # 전체 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r == 1]
    A2_mean_response_time = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'A2_response_rate': A2_response_rate,
        'A2_slow_to_fast_rt_diff': A2_slow_to_fast_rt_diff,
        'A2_correct_ratio_by_speed': A2_correct_ratio_by_speed,
        'A2_mean_response_time': A2_mean_response_time
    })

def compute_A3_features(row):
    arrow_size     = row.get('A3-1_list', np.array([]))
    arrow_pos      = row.get('A3-2_list', np.array([]))
    arrow_dir      = row.get('A3-3_list', np.array([]))
    correct_pos    = row.get('A3-4_list', np.array([]))
    resp_type      = row.get('A3-5_list', np.array([]))
    resp           = row.get('A3-6_list', np.array([]))
    rt             = row.get('A3-7_list', np.array([]))

    L = min(len(arrow_size), len(arrow_pos), len(arrow_dir), len(correct_pos), len(resp_type), len(resp), len(rt))
    if L == 0:
        return pd.Series({
            'A3_valid_accuracy': np.nan,
            'A3_invalid_accuracy': np.nan,
            'A3_total_accuracy': np.nan,
            'A3_valid_rt': np.nan,
            'A3_invalid_rt': np.nan,
            'A3_correct_rt': np.nan,
            'A3_incorrect_rt': np.nan,
            'A3_accuracy_gap': np.nan
        })

    # 배열로 변환
    arrow_size, arrow_pos, arrow_dir = np.array(arrow_size[:L]), np.array(arrow_pos[:L]), np.array(arrow_dir[:L])
    correct_pos, resp_type, resp, rt = np.array(correct_pos[:L]), np.array(resp_type[:L]), np.array(resp[:L]), np.array(rt[:L])

    # valid / invalid trial 인덱스 (예: 1=valid, 3=invalid)
    valid_idx = np.where(resp_type == 1)[0]
    invalid_idx = np.where(resp_type == 3)[0]

    # 정확도 계산
    A3_valid_accuracy = (resp[valid_idx] == 1).mean() if len(valid_idx) > 0 else np.nan
    A3_invalid_accuracy = (resp[invalid_idx] == 1).mean() if len(invalid_idx) > 0 else np.nan
    A3_total_accuracy = (resp == 1).mean() if len(resp) > 0 else np.nan

    # 반응시간 계산 (응답한 trial만)
    A3_valid_rt = rt[valid_idx].mean() if len(valid_idx) > 0 else np.nan
    A3_invalid_rt = rt[invalid_idx].mean() if len(invalid_idx) > 0 else np.nan
    A3_correct_rt = rt[resp == 1].mean() if np.any(resp == 1) else np.nan
    A3_incorrect_rt = rt[resp == 0].mean() if np.any(resp == 0) else np.nan

    # valid / invalid 정확도 차이
    if not np.isnan(A3_valid_accuracy) and not np.isnan(A3_invalid_accuracy):
        A3_accuracy_gap = A3_valid_accuracy - A3_invalid_accuracy
    else:
        A3_accuracy_gap = np.nan

    return pd.Series({
        'A3_valid_accuracy': A3_valid_accuracy,
        'A3_invalid_accuracy': A3_invalid_accuracy,
        'A3_total_accuracy': A3_total_accuracy,
        'A3_valid_rt': A3_valid_rt,
        'A3_invalid_rt': A3_invalid_rt,
        'A3_correct_rt': A3_correct_rt,
        'A3_incorrect_rt': A3_incorrect_rt,
        'A3_accuracy_gap': A3_accuracy_gap
    })

def compute_A4_features(row):
    condition = row.get('A4-1_list', np.array([]))
    resp1     = row.get('A4-2_list', np.array([]))
    resp2     = row.get('A4-3_list', np.array([]))
    rt        = row.get('A4-4_list', np.array([]))

    L = min(len(condition), len(resp1), len(resp2), len(rt))
    if L == 0:
        return pd.Series({
            'A4_congruent_accuracy': np.nan,
            'A4_incongruent_accuracy': np.nan,
            'A4_accuracy_gap': np.nan,
            'A4_mean_rt_con': np.nan,
            'A4_mean_rt_incon': np.nan,
            'A4_rt_gap': np.nan,
            'A4_response_rate': 0
        })

    condition, resp1, resp2, rt = np.array(condition[:L]), np.array(resp1[:L]), np.array(resp2[:L]), np.array(rt[:L])

    # 응답이 있는 trial만
    valid_idx = np.where((resp1 != -1) & (resp2 != -1))[0]  # -1 등으로 결측 없음 가정
    response_rate = len(valid_idx)/L if L>0 else 0

    # congruent / incongruent trial
    con_idx = valid_idx[condition[valid_idx] == 1]
    incon_idx = valid_idx[condition[valid_idx] == 2]

    # 정확도 계산 (resp1==1이 정답)
    con_acc = (resp1[con_idx] == 1).mean() if len(con_idx) > 0 else np.nan
    incon_acc = (resp1[incon_idx] == 1).mean() if len(incon_idx) > 0 else np.nan
    acc_gap = con_acc - incon_acc if not np.isnan(con_acc) and not np.isnan(incon_acc) else np.nan

    # 반응시간 계산
    mean_rt_con = rt[con_idx].mean() if len(con_idx) > 0 else np.nan
    mean_rt_incon = rt[incon_idx].mean() if len(incon_idx) > 0 else np.nan
    rt_gap = mean_rt_incon - mean_rt_con if not np.isnan(mean_rt_con) and not np.isnan(mean_rt_incon) else np.nan

    return pd.Series({
        'A4_congruent_accuracy': con_acc,
        'A4_incongruent_accuracy': incon_acc,
        'A4_accuracy_gap': acc_gap,
        'A4_mean_rt_con': mean_rt_con,
        'A4_mean_rt_incon': mean_rt_incon,
        'A4_rt_gap': rt_gap,
        'A4_response_rate': response_rate
    })

def compute_A5_features(row):
    change_type = row.get('A5-1_list', np.array([]))
    resp1       = row.get('A5-2_list', np.array([]))
    resp2       = row.get('A5-3_list', np.array([]))
    L = min(len(change_type), len(resp1))
    if L == 0:
        return pd.Series({
            'A5_accuracy_non_change': np.nan,
            'A5_accuracy_pos_change': np.nan,
            'A5_accuracy_color_change': np.nan,
            'A5_accuracy_shape_change': np.nan,
            'A5_accuracy_var': np.nan
        })

    change_type = np.array(change_type[:L])
    resp1 = np.array(resp1[:L])

    # 각 조건별 인덱스
    idx_non_change  = np.where(change_type == 1)[0]
    idx_pos_change  = np.where(change_type == 2)[0]
    idx_color_change = np.where(change_type == 3)[0]
    idx_shape_change = np.where(change_type == 4)[0]

    # 정확도 계산 (1=정답)
    acc_non = (resp1[idx_non_change] == 1).mean() if len(idx_non_change) > 0 else np.nan
    acc_pos = (resp1[idx_pos_change] == 1).mean() if len(idx_pos_change) > 0 else np.nan
    acc_color = (resp1[idx_color_change] == 1).mean() if len(idx_color_change) > 0 else np.nan
    acc_shape = (resp1[idx_shape_change] == 1).mean() if len(idx_shape_change) > 0 else np.nan

    # 변화 유형 간 정확도 분산
    acc_list = [acc_non, acc_pos, acc_color, acc_shape]
    acc_var = np.nanvar(acc_list)  # NaN 자동 무시

    return pd.Series({
        'A5_accuracy_non_change': acc_non,
        'A5_accuracy_pos_change': acc_pos,
        'A5_accuracy_color_change': acc_color,
        'A5_accuracy_shape_change': acc_shape,
        'A5_accuracy_var': acc_var
    })

In [8]:
train_a = preprocess_A(train_a) if len(train_a) else pd.DataFrame()
train_a.info()

Step 1: Age 파생...
Step 2: Sequence 변환...
Step 3: A1 feature 생성...
Step 4: A2 feature 생성...
Step 5: A3 feature 생성...
Step 6: A4 feature 생성...
Step 7: A5 feature 생성...
Step 8: A6 feature 생성...
Step 9: A7 feature 생성...
Step 10: A8 feature 생성...
Step 9: A7 feature 생성...
A 검사 데이터 전처리 완료
<class 'pandas.core.frame.DataFrame'>
Index: 647237 entries, 0 to 647240
Data columns (total 51 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Test_id                    647237 non-null  object 
 1   Test_x                     647237 non-null  object 
 2   Label                      647237 non-null  int64  
 3   PrimaryKey                 647237 non-null  object 
 4   Age                        647237 non-null  float64
 5   TestDate                   647237 non-null  int64  
 6   A1_response_rate           647237 non-null  float64
 7   A1_left_response_rate      647237 non-null  float64
 8   A1_right_response_rate     647237 no

## **Model**

In [17]:
x = train_a.drop(columns=['Label', 'Test_id', 'Test_x', 'PrimaryKey', 'TestDate'])
y = train_a['Label']

x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)

smote = SMOTE(random_state=42)
x_train_res, y_train_res = smote.fit_resample(x_train, y_train)

# 스케일러 학습
scaler = StandardScaler()
x_train_res = scaler.fit_transform(x_train_res)
x_val = scaler.transform(x_val)

# 스케일러 저장
joblib.dump(scaler, "./scaler_A.pkl")
print("✅ Scaler saved as './scaler_A.pkl'")

def objective_dnn(trial):
    n_layers = trial.suggest_int("n_layers", 1, 12)
    units = trial.suggest_categorical("units", [32, 64, 128, 256])
    dropout_rate = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    model = Sequential()
    model.add(Dense(units, input_dim=x_train_res.shape[1], activation='relu'))
    model.add(BatchNormalization())
    if dropout_rate > 0:
        model.add(Dropout(dropout_rate))

    for _ in range(n_layers - 1):
        model.add(Dense(units, activation='relu'))
        model.add(BatchNormalization())
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer=Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])

    es_callbacks = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(
        x_train_res, y_train_res,
        validation_data=(x_val, y_val),
        epochs=50,
        batch_size=batch_size,
        callbacks=[es_callbacks],
        verbose=0
    )

    preds = model.predict(x_val).ravel()
    return roc_auc_score(y_val, preds)

study_dnn = optuna.create_study(direction="maximize")
study_dnn.optimize(objective_dnn, n_trials=30)

# 최적 하이퍼파라미터로 DNN 모델 생성
dnn_model = Sequential()
n_layers = study_dnn.best_params['n_layers']
units = study_dnn.best_params['units']
dropout_rate = study_dnn.best_params['dropout']
lr = study_dnn.best_params['lr']

dnn_model.add(Dense(units, input_dim=x_train_res.shape[1], activation='relu'))
dnn_model.add(BatchNormalization())
if dropout_rate > 0:
    dnn_model.add(Dropout(dropout_rate))

for _ in range(n_layers - 1):
    dnn_model.add(Dense(units, activation='relu'))
    dnn_model.add(BatchNormalization())
    if dropout_rate > 0:
        dnn_model.add(Dropout(dropout_rate))
dnn_model.add(Dense(1, activation='sigmoid'))

dnn_model.compile(optimizer=Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = dnn_model.fit(
    x_train_res, y_train_res,
    validation_data=(x_val, y_val),
    epochs=500,
    batch_size=study_dnn.best_params['batch_size'],
    callbacks=[early_stop],
    verbose=1
)

# 학습 완료 모델 저장
save_model(dnn_model, "./dnn_model_A.h5")
print("✅ DNN model saved as './dnn_model_A.h5'")

[I 2025-11-11 10:07:02,126] A new study created in memory with name: no-name-145fc8ea-a91e-4c0e-b34f-12305ef76588


✅ Scaler saved as '.scaler_A.pkl'
4046/4046 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step


[I 2025-11-11 10:09:13,010] Trial 0 finished with value: 0.6442617940426012 and parameters: {'n_layers': 3, 'units': 64, 'dropout': 0.3101217587806867, 'lr': 0.0021159081960068836, 'batch_size': 256}. Best is trial 0 with value: 0.6442617940426012.


4046/4046 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step


[I 2025-11-11 10:13:36,125] Trial 1 finished with value: 0.5826245207047942 and parameters: {'n_layers': 11, 'units': 32, 'dropout': 0.3846725415877498, 'lr': 0.00016688331660623723, 'batch_size': 256}. Best is trial 0 with value: 0.6442617940426012.


...

[I 2025-11-11 11:59:25,054] Trial 28 finished with value: 0.6484098233569925 and parameters: {'n_layers': 7, 'units': 64, 'dropout': 0.2104110667547521, 'lr': 0.00982016342244242, 'batch_size': 256}. Best is trial 17 with value: 0.6519503566420701.


4046/4046 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step


[I 2025-11-11 12:02:00,229] Trial 29 finished with value: 0.6451318878028298 and parameters: {'n_layers': 6, 'units': 64, 'dropout': 0.08257764155048525, 'lr': 0.006098437937593239, 'batch_size': 256}. Best is trial 17 with value: 0.6519503566420701.


Epoch 1/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 27s 5ms/step - accuracy: 0.8792 - loss: 0.2618 - val_accuracy: 0.9766 - val_loss: 0.1083
Epoch 2/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9764 - loss: 0.0866 - val_accuracy: 0.9769 - val_loss: 0.1066
Epoch 3/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9790 - loss: 0.0789 - val_accuracy: 0.9767 - val_loss: 0.1089
Epoch 4/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9804 - loss: 0.0756 - val_accuracy: 0.9767 - val_loss: 0.1081
Epoch 5/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9813 - loss: 0.0736 - val_accuracy: 0.9772 - val_loss: 0.1056
Epoch 6/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9821 - loss: 0.0710 - val_accuracy: 0.9759 - val_loss: 0.1078
Epoch 7/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9825 - loss: 0.0705 - val_accuracy: 0.9722 - val_loss: 0.1143
Epoch 8/500
3954/3954 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9828 -

✅ DNN model saved as './dnn_model_A.h5'


# **Test B**

In [26]:
train_raw_b = pd.read_csv('./train/B.csv')
train_label = pd.read_csv('./train.csv')

In [27]:
train_label_b = train_label[train_label['Test'] == 'B']

train_b = train_label_b.merge(train_raw_b, on='Test_id', how='left')

train_b = train_b.drop('Test_y', axis=1)
train_b.rename(columns={'Test_x': 'Test'})
train_b = train_b.dropna()

train_b.info()

<class 'pandas.core.frame.DataFrame'>
Index: 297513 entries, 0 to 297525
Data columns (total 32 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Test_id     297513 non-null  object
 1   Test_x      297513 non-null  object
 2   Label       297513 non-null  int64 
 3   PrimaryKey  297513 non-null  object
 4   Age         297513 non-null  object
 5   TestDate    297513 non-null  int64 
 6   B1-1        297513 non-null  object
 7   B1-2        297513 non-null  object
 8   B1-3        297513 non-null  object
 9   B2-1        297513 non-null  object
 10  B2-2        297513 non-null  object
 11  B2-3        297513 non-null  object
 12  B3-1        297513 non-null  object
 13  B3-2        297513 non-null  object
 14  B4-1        297513 non-null  object
 15  B4-2        297513 non-null  object
 16  B5-1        297513 non-null  object
 17  B5-2        297513 non-null  object
 18  B6          297513 non-null  object
 19  B7          297513 non-null 

## **Features**

In [28]:
def safe_fromstring(x, dtype=float):
    if isinstance(x, str) and x.strip():
        return np.fromstring(x, sep=',', dtype=dtype)
    return np.array([], dtype=dtype)

def preprocess_B(train_B):
    df = train_B.copy()

    print("Step 1: Age 파생...")
    df["Age"] = df["Age"].astype(str).str.extract(r'(\d+)').astype(float)
    feats = pd.DataFrame(index=df.index)

    print("Step 2: Sequence 변환...")
    seq_cols = [
        "B1-1","B1-2","B1-3",
        "B2-1","B2-2","B2-3",
        "B3-1","B3-2",
        "B4-1","B4-2",
        "B5-1","B5-2",
        "B6","B7","B8"
    ]
    for col in seq_cols:
        feats[col + '_list'] = df[col].apply(lambda x: safe_fromstring(x, dtype=float))

    print("Step 3: B1 feature 생성...")
    b1_feats = feats.apply(compute_B1_features, axis=1)
    feats = pd.concat([feats, b1_feats], axis=1)

    print("Step 4: B2 feature 생성...")
    b2_feats = feats.apply(compute_B2_features, axis=1)
    feats = pd.concat([feats, b2_feats], axis=1)

    print("Step 5: B3 feature 생성...")
    b3_feats = feats.apply(compute_B3_features, axis=1)
    feats = pd.concat([feats, b3_feats], axis=1)

    print("Step 6: B4 feature 생성...")
    b4_feats = feats.apply(compute_B4_features, axis=1)
    feats = pd.concat([feats, b4_feats], axis=1)

    print("Step 7: B5 feature 생성...")
    b5_feats = feats.apply(compute_B5_features, axis=1)
    feats = pd.concat([feats, b5_feats], axis=1)

    print("Step 8: B6 feature 생성...")
    b6_feats = feats.apply(compute_B6_features, axis=1)
    feats = pd.concat([feats, b6_feats], axis=1)

    print("Step 9: B7 feature 생성...")
    b7_feats = feats.apply(compute_B7_features, axis=1)
    feats = pd.concat([feats, b7_feats], axis=1)

    print("Step 10: B8 feature 생성...")
    b8_feats = feats.apply(compute_B8_features, axis=1)
    feats = pd.concat([feats, b8_feats], axis=1)

    print("Step 10: B8 feature 생성...")
    feats['B9_aud_hit'] = df['B9-1']
    feats['B9_aud_miss'] = df['B9-2']
    feats['B9_aud_fa'] = df['B9-3']
    feats['B9_aud_cr'] = df['B9-4']
    feats['B9_vis_err'] = df['B9-5']

    print("Step 10: B8 feature 생성...")
    feats['B10_aud_hit'] = df['B10-1']
    feats['B10_aud_miss'] = df['B10-2']
    feats['B10_aud_fa'] = df['B10-3']
    feats['B10_aud_cr'] = df['B10-4']
    feats['B10_vis1_err'] = df['B10-5']
    feats['B10_vis2_correct'] = df['B10-6']

    feats = feats.fillna(0)

    print("B 검사 데이터 전처리 완료")
    list_cols = [f'{cols}_list' for cols in seq_cols]
    int_cols = [
        'B9-1', 'B9-2', 'B9-3', 'B9-4', 'B9-5',
        'B10-1', 'B10-2', 'B10-3', 'B10-4', 'B10-5', 'B10-6'
    ]
    out = pd.concat([df.drop(columns=seq_cols + int_cols, errors="ignore"),
                     feats.drop(columns=list_cols, errors='ignore')], axis=1)
    return out


In [29]:
def compute_B1_features(row):
    r1 = row.get('B1-1_list', np.array([]))
    rt = row.get('B1-2_list', np.array([]))
    r2 = row.get('B1-3_list', np.array([]))

    L = min(len(r1), len(rt), len(r2))
    if L == 0:
        return pd.Series({
            'B1_task1_accuracy': 0,
            'B1_task2_change_acc': 0,
            'B1_task2_non_change_acc': 0,
            'B1_task2_accuracy_gap': 0,
            'B1_task2_mean_rt': np.nan
        })

    r1, r2, rt = np.array(r1[:L]), np.array(r2[:L]), np.array(rt[:L])

    # 1과제 정답률
    r1_bin = np.array([1 if val == 1 else 0 for val in r1])
    B1_task1_accuracy = r1_bin.mean()

    # 2과제: change / non-change 정확도
    r2_bin = np.array([1 if val == 1 else 0 for val in r2])
    change_mask = np.arange(L) < L//2        # 앞 절반이 change
    non_change_mask = np.arange(L) >= L//2   # 뒤 절반이 non-change

    B1_task2_change_acc = r2_bin[change_mask].mean() if np.any(change_mask) else 0
    B1_task2_non_change_acc = r2_bin[non_change_mask].mean() if np.any(non_change_mask) else 0

    # 정확도 차
    B1_task2_accuracy_gap = B1_task2_change_acc - B1_task2_non_change_acc

    # 2과제 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r2_bin == 1]
    B1_task2_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B1_task1_accuracy': B1_task1_accuracy,
        'B1_task2_change_acc': B1_task2_change_acc,
        'B1_task2_non_change_acc': B1_task2_non_change_acc,
        'B1_task2_accuracy_gap': B1_task2_accuracy_gap,
        'B1_task2_mean_rt': B1_task2_mean_rt
    })

def compute_B2_features(row):
    r1 = row.get('B2-1_list', np.array([]))
    rt = row.get('B2-2_list', np.array([]))
    r2 = row.get('B2-3_list', np.array([]))

    L = min(len(r1), len(rt), len(r2))
    if L == 0:
        return pd.Series({
            'B2_task1_accuracy': 0,
            'B2_task2_change_acc': 0,
            'B2_task2_non_change_acc': 0,
            'B2_task2_accuracy_gap': 0,
            'B2_task2_mean_rt': np.nan
        })

    r1, r2, rt = np.array(r1[:L]), np.array(r2[:L]), np.array(rt[:L])

    # 1과제 정답률
    r1_bin = np.array([1 if val == 1 else 0 for val in r1])
    B2_task1_accuracy = r1_bin.mean()

    # 2과제: change / non-change 정확도
    r2_bin = np.array([1 if val == 1 else 0 for val in r2])
    change_mask = np.arange(L) < L//2
    non_change_mask = np.arange(L) >= L//2

    B2_task2_change_acc = r2_bin[change_mask].mean() if np.any(change_mask) else 0
    B2_task2_non_change_acc = r2_bin[non_change_mask].mean() if np.any(non_change_mask) else 0

    # 정확도 차
    B2_task2_accuracy_gap = B2_task2_change_acc - B2_task2_non_change_acc

    # 2과제 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r2_bin == 1]
    B2_task2_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B2_task1_accuracy': B2_task1_accuracy,
        'B2_task2_change_acc': B2_task2_change_acc,
        'B2_task2_non_change_acc': B2_task2_non_change_acc,
        'B2_task2_accuracy_gap': B2_task2_accuracy_gap,
        'B2_task2_mean_rt': B2_task2_mean_rt
    })

def compute_B3_features(row):
    # 리스트 추출
    r = row.get('B3-1_list', np.array([]))
    rt = row.get('B3-2_list', np.array([]))

    L = min(len(r), len(rt))
    if L == 0:
        return pd.Series({
            'B3_accuracy': 0,
            'B3_mean_rt': np.nan
        })

    r, rt = np.array(r[:L]), np.array(rt[:L])

    # 전체 정확도
    B3_accuracy = r.mean()

    # 전체 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r == 1]
    B3_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B3_accuracy': B3_accuracy,
        'B3_mean_rt': B3_mean_rt
    })

def compute_B4_features(row):
    r = row.get('B4-1_list', np.array([]))
    rt = row.get('B4-2_list', np.array([]))

    L = min(len(r), len(rt))
    if L == 0:
        return pd.Series({
            'B4_congruent_accuracy': 0,
            'B4_incongruent_accuracy': 0,
            'B4_accuracy_gap': 0,
            'B4_mean_rt_congruent': np.nan,
            'B4_mean_rt_incongruent': np.nan,
            'B4_rt_gap': np.nan
        })

    r, rt = np.array(r[:L]), np.array(rt[:L])

    # 정답 1, 오답 0 변환
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # mask 생성
    congruent_mask = np.arange(L) < L//2       # 앞 30 trials
    incongruent_mask = np.arange(L) >= L//2    # 뒤 30 trials

    # 정확도
    B4_congruent_accuracy = r_bin[congruent_mask].mean() if np.any(congruent_mask) else 0
    B4_incongruent_accuracy = r_bin[incongruent_mask].mean() if np.any(incongruent_mask) else 0
    B4_accuracy_gap = B4_incongruent_accuracy - B4_congruent_accuracy

    # 평균 반응시간 (응답한 trial만)
    valid_rt_con = rt[congruent_mask & (r_bin == 1)]
    valid_rt_incon = rt[incongruent_mask & (r_bin == 1)]

    B4_mean_rt_congruent = valid_rt_con.mean() if len(valid_rt_con) > 0 else np.nan
    B4_mean_rt_incongruent = valid_rt_incon.mean() if len(valid_rt_incon) > 0 else np.nan
    B4_rt_gap = B4_mean_rt_incongruent - B4_mean_rt_congruent if len(valid_rt_con) > 0 and len(valid_rt_incon) > 0 else np.nan

    return pd.Series({
        'B4_congruent_accuracy': B4_congruent_accuracy,
        'B4_incongruent_accuracy': B4_incongruent_accuracy,
        'B4_accuracy_gap': B4_accuracy_gap,
        'B4_mean_rt_congruent': B4_mean_rt_congruent,
        'B4_mean_rt_incongruent': B4_mean_rt_incongruent,
        'B4_rt_gap': B4_rt_gap
    })

def compute_B5_features(row):
    r = row.get('B5-1_list', np.array([]))
    rt = row.get('B5-2_list', np.array([]))

    L = min(len(r), len(rt))
    if L == 0:
        return pd.Series({
            'B5_accuracy': 0,
            'B5_mean_rt': np.nan
        })

    r, rt = np.array(r[:L]), np.array(rt[:L])

    # 정답 1, 오답 0 변환
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B5_accuracy = r_bin.mean()

    # 평균 반응시간 (응답한 trial만)
    valid_rt = rt[r_bin == 1]
    B5_mean_rt = valid_rt.mean() if len(valid_rt) > 0 else np.nan

    return pd.Series({
        'B5_accuracy': B5_accuracy,
        'B5_mean_rt': B5_mean_rt
    })

def compute_B6_features(row):
    r = row.get('B6_list', np.array([]))

    if len(r) == 0:
        return pd.Series({'B6_accuracy': 0})

    # 정답 1, 오답 0
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B6_accuracy = r_bin.mean()

    return pd.Series({'B6_accuracy': B6_accuracy})

def compute_B7_features(row):
    r = row.get('B7_list', np.array([]))

    if len(r) == 0:
        return pd.Series({'B7_accuracy': 0})

    # 정답 1, 오답 0
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B7_accuracy = r_bin.mean()

    return pd.Series({'B7_accuracy': B7_accuracy})

def compute_B8_features(row):
    r = row.get('B8_list', np.array([]))

    if len(r) == 0:
        return pd.Series({'B8_accuracy': 0})

    # 정답 1, 오답 0
    r_bin = np.array([1 if val == 1 else 0 for val in r])

    # 전체 정확도
    B8_accuracy = r_bin.mean()

    return pd.Series({'B8_accuracy': B8_accuracy})

In [30]:
train_b = preprocess_B(train_b) if len(train_b) else pd.DataFrame()
train_b.info()

Step 1: Age 파생...
Step 2: Sequence 변환...
Step 3: B1 feature 생성...
Step 4: B2 feature 생성...
Step 5: B3 feature 생성...
Step 6: B4 feature 생성...
Step 7: B5 feature 생성...
Step 8: B6 feature 생성...
Step 9: B7 feature 생성...
Step 10: B8 feature 생성...
Step 10: B8 feature 생성...
Step 10: B8 feature 생성...
B 검사 데이터 전처리 완료
<class 'pandas.core.frame.DataFrame'>
Index: 297513 entries, 0 to 297525
Data columns (total 40 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Test_id                  297513 non-null  object 
 1   Test_x                   297513 non-null  object 
 2   Label                    297513 non-null  int64  
 3   PrimaryKey               297513 non-null  object 
 4   Age                      297513 non-null  float64
 5   TestDate                 297513 non-null  int64  
 6   B1_task1_accuracy        297513 non-null  float64
 7   B1_task2_change_acc      297513 non-null  float64
 8   B1_task2_non_change_acc  2975

## **Model**

In [31]:
x = train_b.drop(columns=['Label', 'Test_id', 'Test_x', 'PrimaryKey', 'TestDate'])
y = train_b['Label']

x_train, x_val, y_train, y_val = train_test_split(
    x, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

smote = SMOTE(random_state=42)
x_train_res, y_train_res = smote.fit_resample(x_train, y_train)

def objective_lgb_auc(trial):
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
    }

    model = LGBMClassifier(
        **param,
        n_jobs=-1,
        force_col_wise=True,
        verbosity=-1
    )
    model.fit(x_train_res, y_train_res)

    preds_proba = model.predict_proba(x_val)[:, 1]
    return roc_auc_score(y_val, preds_proba)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb_auc, n_trials=30)

print("Best LGBM params:", study_lgb.best_params)

best_lgb = LGBMClassifier(
    **study_lgb.best_params,
    n_jobs=-1,
    force_col_wise=True,
    verbosity=-1
)

best_lgb.fit(x_train_res, y_train_res)

joblib.dump(best_lgb, "./lgbm_B.pkl")
print("✅ LGBM model saved as './lgbm_B.pkl'")

[I 2025-11-11 13:32:15,788] A new study created in memory with name: no-name-3062942b-bacc-413d-88fa-37a0b380fdfe
[I 2025-11-11 13:32:26,167] Trial 0 finished with value: 0.5048359123253524 and parameters: {'n_estimators': 456, 'max_depth': 9, 'learning_rate': 0.1701211918073389, 'num_leaves': 109, 'subsample': 0.7008430446972783, 'colsample_bytree': 0.697137643744375}. Best is trial 0 with value: 0.5048359123253524.
[I 2025-11-11 13:32:35,050] Trial 1 finished with value: 0.537907482028317 and parameters: {'n_estimators': 240, 'max_depth': 15, 'learning_rate': 0.013979959162604888, 'num_leaves': 64, 'subsample': 0.5301603151855685, 'colsample_bytree': 0.7403360479963528}. Best is trial 1 with value: 0.537907482028317.
[I 2025-11-11 13:32:38,642] Trial 2 finished with value: 0.5154152608893425 and parameters: {'n_estimators': 110, 'max_depth': 11, 'learning_rate': 0.019574543428386337, 'num_leaves': 88, 'subsample': 0.5010075265716603, 'colsample_bytree': 0.78236180730133}. Best is tri

Best LGBM params: {'n_estimators': 981, 'max_depth': 15, 'learning_rate': 0.012932002248895942, 'num_leaves': 76, 'subsample': 0.7030037968873075, 'colsample_bytree': 0.533970928780066}
✅ LGBM model saved as './lgbm_B.pkl'
